In [4]:
"""
generate_ar_gif.py  (v3 — matches diagram)
===========================================
Visual style closely follows the architecture figure:
  • Bottom-to-top data flow
  • Dashed outer encoder / decoder boxes
  • Purple / lavender encoder output patches
  • Causal self-attn drawn as two token rows + diagonal arrows (box-in-box)
  • Residual skip arrows on the right side of each module
  • Vertical decoder input stack (start token + growing predictions)
  • Fan of causal cross-attention arrows at the very top
  • Feedback dashed path on the right

LLM variant shows word tokens; same overall layout, horizontal decoder.

pip install matplotlib pillow
python generate_ar_gif.py
→ llm_ar.gif   tsfm_ar.gif   combined_ar.gif
"""

import io
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image

In [6]:
"""
generate_ar_gif.py  (v4)
========================
pip install matplotlib pillow
python generate_ar_gif.py  →  tsfm_ar.gif   llm_ar.gif   combined_ar.gif

Matches the architecture diagram:
  • Bottom-to-top flow
  • Dashed encoder / decoder module borders
  • Purple encoder output patches
  • Two-row causal self-attn inside encoder
  • Vertical decoder input stack
  • Fan cross-attention arrows at top
"""

import io
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image

# ─── Palette ──────────────────────────────────────────────────────────────
NAVY   = '#1e3a5c'
NAVY2  = '#0d2240'
PURP   = '#7b6ab4'    # lavender – encoder output patches
PURP2  = '#5a4a95'
MINT   = '#dff0e6'
MINTB  = '#7aaa8a'
RED    = '#cc3333'
DARK   = '#222222'
WHITE  = 'white'
MODBG  = '#f3faf5'    # very-light inside dashed module

# ─── Figure  (24 × 13 in, 90 dpi → 2160 × 1170 px) ──────────────────────
FW, FH, DPI = 24.0, 13.0, 90

# Column bounds
EX0, EX1 = 0.6,  10.4   # encoder x
DX0, DX1 = 13.6, 23.4   # decoder x
ECX = (EX0 + EX1) / 2    # 5.5
DCX = (DX0 + DX1) / 2    # 18.5
EW  = EX1 - EX0          # 9.8
DW  = DX1 - DX0          # 9.8

# ─── Token geometry ───────────────────────────────────────────────────────
HTW, HTH, HTGAP = 0.96, 0.64, 0.24   # horizontal token
VTW, VTH, VTGAP = 1.20, 0.58, 0.14   # vertical stack token

# ─── Encoder layout (y, bottom → top) ────────────────────────────────────
E_IN_Y    = 1.30
PH        = 0.72   # pipeline block height (no gap – stacked touching)
E_PIPE_Y0 = E_IN_Y + HTH/2 + 0.26    # 1.88
E_PIPE_N  = 4
E_PIPE_TOP = E_PIPE_Y0 + E_PIPE_N * PH   # 4.76
E_MOD_Y0  = E_PIPE_TOP + 0.26        # 5.02
E_MOD_H   = 5.50
E_MOD_Y1  = E_MOD_Y0 + E_MOD_H      # 10.52
E_OUT_Y   = E_MOD_Y1 + 0.44         # 10.96

# Inside encoder module
ESA_PAD   = 0.24                     # padding inside module before self-attn box
ESA_Y0    = E_MOD_Y0 + ESA_PAD
ESA_H     = 2.30
ESA_Y1    = ESA_Y0 + ESA_H
ESA_BOT   = ESA_Y0 + 0.30           # bottom token row
ESA_TOP   = ESA_Y1 - 0.30           # top token row

IRH = 0.46   # inner-row height
IRG = 0.18   # inner-row gap
IR_Y0 = ESA_Y1 + 0.36              # first inner row bottom

# ─── TSFM Decoder layout ─────────────────────────────────────────────────
D_STACK_TOP  = 3.60   # y-center of TOP (most recently added) token in stack
D_SLOT       = VTH + VTGAP            # 0.72
D_STACK_MAX  = 5

D_PIPE_Y0  = D_STACK_TOP + VTH/2 + 0.24    # 4.13
D_PIPE_N   = 2
D_PIPE_TOP = D_PIPE_Y0 + D_PIPE_N * PH     # 5.57
D_MOD_Y0   = D_PIPE_TOP + 0.26             # 5.83
D_MOD_H    = 4.40
D_MOD_Y1   = D_MOD_Y0 + D_MOD_H           # 10.23
HEAD_Y0    = D_MOD_Y1 + 0.26              # 10.49
HEAD_H     = 0.72
HEAD_Y1    = HEAD_Y0 + HEAD_H             # 11.21
D_PRED_TOP = HEAD_Y1 + 0.30              # 11.51  (bottom of FIRST output pred)

# ─── LLM Decoder layout ───────────────────────────────────────────────────
# Mirrors encoder (same y-ranges, different content)
LLM_MOD_Y0 = E_MOD_Y0
LLM_MOD_H  = E_MOD_H - 0.20
LLM_MOD_Y1 = LLM_MOD_Y0 + LLM_MOD_H
LLM_HEAD_Y0 = LLM_MOD_Y1 + 0.26
LLM_HEAD_Y1 = LLM_HEAD_Y0 + HEAD_H
LLM_OUT_Y   = LLM_HEAD_Y1 + 0.44


# ══════════════════════════════════════════════════════════════════════════
#  Drawing primitives  (all font sizes ≥ 12)
# ══════════════════════════════════════════════════════════════════════════

def new_fig():
    fig, ax = plt.subplots(figsize=(FW, FH), dpi=DPI)
    fig.patch.set_facecolor(WHITE); ax.set_facecolor(WHITE)
    ax.set_xlim(0, FW); ax.set_ylim(0, FH)
    ax.set_aspect('equal'); ax.axis('off')
    return fig, ax


def hc(n, cx, tw=HTW, gap=HTGAP):
    """Horizontal token x-centers."""
    tot = n*tw + (n-1)*gap
    x0  = cx - tot/2 + tw/2
    return [x0 + i*(tw+gap) for i in range(n)]


def htok(ax, cx, cy, label, style='navy', fresh=False, alpha=1.0,
         tw=HTW, th=HTH):
    """Draw one horizontal token."""
    if style == 'navy':
        fc,ec,lw,ls = NAVY,(('#f0a000' if fresh else NAVY2)),(2.2 if fresh else 1.0),'solid'; tc=WHITE
    elif style == 'purple':
        fc,ec,lw,ls = PURP,PURP2,1.0,'solid'; tc=WHITE
    elif style == 'pred':
        fc,ec,lw,ls = WHITE,RED,1.8,(0,(5,3)); tc=RED
    elif style == 'word':
        fc,ec,lw,ls = NAVY,(('#f0a000' if fresh else NAVY2)),(2.2 if fresh else 1.0),'solid'; tc=WHITE
    else:
        fc,ec,lw,ls = WHITE,DARK,1.0,'solid'; tc=DARK

    ax.add_patch(FancyBboxPatch(
        (cx-tw/2, cy-th/2), tw, th,
        boxstyle='round,pad=0.06',
        facecolor=fc, edgecolor=ec, linewidth=lw, linestyle=ls,
        zorder=3, alpha=alpha))
    fs = 11 if len(label)>4 else (12 if len(label)>2 else 13)
    ax.text(cx, cy, label, ha='center', va='center',
            fontsize=fs, color=tc, fontweight='bold',
            fontfamily='monospace', zorder=4, alpha=alpha)


def vtok(ax, cx, cy, label, style='navy', fresh=False, alpha=1.0):
    htok(ax, cx, cy, label, style=style, fresh=fresh, alpha=alpha,
         tw=VTW, th=VTH)


def pipe_blk(ax, x0, y0, w, h, label):
    ax.add_patch(FancyBboxPatch(
        (x0,y0), w, h, boxstyle='round,pad=0.06',
        facecolor=MINT, edgecolor=MINTB, linewidth=1.0, zorder=2))
    ax.text(x0+w/2, y0+h/2, label, ha='center', va='center',
            fontsize=13, color='#1a5530', fontweight='bold', zorder=3)


def mod_box(ax, x0, y0, w, h, label, nx=None):
    """Dashed outer module box."""
    ax.add_patch(FancyBboxPatch(
        (x0,y0), w, h, boxstyle='round,pad=0.10',
        facecolor=MODBG, edgecolor='#888888',
        linewidth=1.4, linestyle=(0,(7,4)), zorder=2))
    if nx:
        ax.text(x0+w-0.14, y0+h-0.14, f'n \u00d7',
                ha='right', va='top', fontsize=14,
                color='#777', style='italic', zorder=5)
    ax.text(x0+w-0.18, y0+0.18, label,
            ha='right', va='bottom', fontsize=14,
            color='#1a5530', fontweight='bold', style='italic', zorder=5)


def irow(ax, x0, y0, w, h, label):
    """White rounded inner-row box."""
    ax.add_patch(FancyBboxPatch(
        (x0,y0), w, h, boxstyle='round,pad=0.06',
        facecolor=WHITE, edgecolor='#aaaaaa', linewidth=0.9, zorder=4))
    ax.text(x0+w/2, y0+h/2, label, ha='center', va='center',
            fontsize=12, color='#111', zorder=5)


def sa_box(ax, x0, y0, w, h, label):
    """Self-attention inner box."""
    ax.add_patch(FancyBboxPatch(
        (x0,y0), w, h, boxstyle='round,pad=0.08',
        facecolor='#eaf4ea', edgecolor='#888', linewidth=1.0, zorder=3))
    ax.text(x0+w-0.12, y0+h/2, label, ha='right', va='center',
            fontsize=11.5, color='#444', style='italic', zorder=5)


def arr(ax, x0, y0, x1, y1, c=DARK, lw=1.2, ls='solid',
        arc=0.0, zorder=3, alpha=1.0, hw=0.13, hl=0.13):
    ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
        arrowprops=dict(
            arrowstyle=f'->, head_width={hw}, head_length={hl}',
            color=c, lw=lw, linestyle=ls,
            connectionstyle=f'arc3,rad={arc}'),
        zorder=zorder, alpha=alpha)


def dline(ax, xs, ys, c=RED, lw=1.2):
    ax.plot(xs, ys, color=c, lw=lw, linestyle=(0,(5,3)), zorder=3)


def skip_arrow(ax, x, y_top, y_bot):
    """Dashed residual-skip arrow on the right side of a module."""
    ax.annotate('', xy=(x, y_top), xytext=(x, y_bot),
        arrowprops=dict(
            arrowstyle='->, head_width=0.09, head_length=0.09',
            color='#888', lw=0.9, linestyle=(0,(3,2)),
            connectionstyle='arc3,rad=-0.50'), zorder=3)


def txt(ax, x, y, s, fs=13, c=DARK, ha='center', va='center', bold=False):
    ax.text(x, y, s, ha=ha, va=va, fontsize=fs,
            color=c, fontweight='bold' if bold else 'normal', zorder=5)


def step_dots(ax, total, current):
    xs = np.linspace(FW/2-0.36*(total-1), FW/2+0.36*(total-1), total)
    for i, x in enumerate(xs):
        c = NAVY if i==current else '#cccccc'
        ax.add_patch(plt.Circle((x, 0.40), 0.12, fc=c, ec=c, zorder=5))


def fig2pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=DPI, facecolor=WHITE)
    buf.seek(0); img = Image.open(buf).copy()
    plt.close(fig); return img


def save_gif(frames, path, dur):
    rgb = [f.convert('RGB') for f in frames]
    rgb[0].save(path, save_all=True, append_images=rgb[1:],
                duration=dur, loop=0, optimize=False)
    print(f'  → {path}')


# ══════════════════════════════════════════════════════════════════════════
#  Encoder (shared)
# ══════════════════════════════════════════════════════════════════════════

def draw_encoder(ax, in_labels, is_llm=False):
    N      = len(in_labels)
    t_sty  = 'word' if is_llm else 'navy'
    o_sty  = 'word' if is_llm else 'purple'
    p_lbls = (['Embedding', 'Positional Encoding']
               if is_llm else
               ['Temporal Norm', 'Tokenization',
                'Projection Layer', 'Positional Encoding'])

    # Input tokens
    in_xs = hc(N, ECX)
    for i, lbl in enumerate(in_labels):
        htok(ax, in_xs[i], E_IN_Y, lbl, style=t_sty)
    if not is_llm:
        txt(ax, EX0-0.14, E_IN_Y,
            r'$x^{(l)} \in \mathbb{R}^{B \times (N+T-1) \times P}$',
            fs=11, c='#888', ha='right')

    arr(ax, ECX, E_IN_Y+HTH/2+0.08, ECX, E_PIPE_Y0, lw=1.2)

    # Pipeline blocks (stacked, touching)
    py = E_PIPE_Y0
    for lbl in p_lbls:
        pipe_blk(ax, EX0+0.12, py, EW-0.24, PH, lbl)
        py += PH
    p_top = py
    if not is_llm:
        txt(ax, EX0-0.14, p_top - PH/2,
            r'$x^{(l)} \in \mathbb{R}^{B \times (N+T-1) \times d}$',
            fs=11, c='#888', ha='right')

    arr(ax, ECX, p_top+0.08, ECX, E_MOD_Y0, lw=1.2)

    # ── Main encoder module ────────────────────────────────────────────────
    mod_box(ax, EX0, E_MOD_Y0, EW, E_MOD_H, 'Encoder', nx=True)

    # Self-attn inner box
    sa_x0 = EX0+0.32; sa_w = EW-0.84
    sa_box(ax, sa_x0, ESA_Y0, sa_w, ESA_H, 'Causal Self\nAttn.')

    # Two token rows inside self-attn
    n_sa  = min(N, 4)
    sa_tw = HTW*0.85; sa_gap = HTGAP*0.75
    sa_xs = hc(n_sa, ECX-0.30, tw=sa_tw, gap=sa_gap)
    for j in range(n_sa):
        lbl = in_labels[j] if is_llm else f'p{j+1}'
        htok(ax, sa_xs[j], ESA_BOT, lbl, style=t_sty, tw=sa_tw, th=HTH*0.85)
        htok(ax, sa_xs[j], ESA_TOP, lbl, style=t_sty, tw=sa_tw, th=HTH*0.85)
        # Causal arrows (bottom → top)
        for i in range(j+1):
            a = max(0.12, 0.75-0.22*(j-i))
            arr(ax, sa_xs[i], ESA_BOT+HTH*0.85/2+0.05,
                    sa_xs[j], ESA_TOP-HTH*0.85/2-0.05,
                    c=NAVY, lw=0.9 if i<j else 1.2,
                    arc=-0.22*(j-i), alpha=a, hw=0.09, hl=0.09)
    if N > 4:
        for yr in [ESA_BOT, ESA_TOP]:
            ax.text(sa_xs[-1]+sa_tw/2+0.14, yr, '\u22ef',
                    ha='left', va='center', fontsize=14, color=NAVY, zorder=4)

    arr(ax, ECX, ESA_Y1+0.04, ECX, IR_Y0, lw=1.0, hw=0.10, hl=0.10)

    # Inner rows: Add & Norm / Feed Forward / Add & Norm
    irw = EW-0.48; ir_x = EX0+0.24
    i_lbls = ['Add & Norm', 'Feed Forward', 'Add & Norm']
    irc = []; ry = IR_Y0
    for k, lbl in enumerate(i_lbls):
        irow(ax, ir_x, ry, irw, IRH, lbl)
        irc.append(ry+IRH/2)
        if k < 2:
            arr(ax, ECX, ry+IRH+0.04, ECX, ry+IRH+IRG-0.04,
                lw=1.0, hw=0.09, hl=0.09)
        ry += IRH+IRG

    # Residual skip arrows (right side)
    rx = EX1 - 0.36
    for yc in [irc[0], irc[2]]:
        skip_arrow(ax, rx, yc+IRH*0.44, yc-0.70)

    arr(ax, ECX, E_MOD_Y1+0.06, ECX, E_OUT_Y-HTH/2-0.08, lw=1.2)

    # Output patches
    out_xs = hc(N, ECX)
    for i, lbl in enumerate(in_labels):
        htok(ax, out_xs[i], E_OUT_Y, lbl, style=o_sty)
    if N > 4:
        ax.text(out_xs[-1]+HTW/2+0.14, E_OUT_Y, '\u22ef',
                ha='left', va='center', fontsize=14,
                color=(PURP if not is_llm else NAVY), zorder=4)
    txt(ax, EX0-0.14, E_OUT_Y,
        r'$\hat{x}^{(l)}$', fs=12, c='#888', ha='right')

    return out_xs


# ══════════════════════════════════════════════════════════════════════════
#  TSFM Decoder
# ══════════════════════════════════════════════════════════════════════════

def draw_decoder_tsfm(ax, dec_toks, nf, gen, enc_out_xs, xattn):
    dec_all  = dec_toks + ([gen] if gen else [])
    stack_cx = DCX - 1.20   # left of centre — leave room for output preds

    # ── Vertical input stack ───────────────────────────────────────────────
    n_show = min(len(dec_all), D_STACK_MAX)
    if n_show > 0:
        sb = D_STACK_TOP - (n_show-1)*D_SLOT - VTH/2 - 0.18
        sh = D_STACK_TOP + VTH/2 + 0.18 - sb
        ax.add_patch(FancyBboxPatch(
            (stack_cx-VTW/2-0.18, sb), VTW+0.36, sh,
            boxstyle='round,pad=0.06',
            facecolor=WHITE, edgecolor='#888', linewidth=1.0,
            linestyle='solid', zorder=2))
        for idx in range(n_show):
            tok = dec_all[idx]
            cy  = D_STACK_TOP - idx*D_SLOT
            pred  = idx >= nf
            fresh = bool(gen) and idx == n_show-1
            vtok(ax, stack_cx, cy, tok,
                 style='pred' if pred else 'navy', fresh=fresh)
    else:
        txt(ax, stack_cx, D_STACK_TOP-1.2, 'start token\nincoming',
            fs=12, c='#aaa')

    txt(ax, DX0-0.14, D_STACK_TOP-(D_STACK_MAX*D_SLOT)/2,
        r'$x^{(l)} \in \mathbb{R}^{B \times T \times P}$',
        fs=11, c='#888', ha='right')

    if dec_all:
        arr(ax, stack_cx, D_STACK_TOP+VTH/2+0.08,
                DCX, D_PIPE_Y0, lw=1.2, arc=-0.18)

    # ── Pipeline blocks ────────────────────────────────────────────────────
    py = D_PIPE_Y0
    for lbl in ['Projection Layer', 'Positional Encoding']:
        pipe_blk(ax, DX0+0.12, py, DW-0.24, PH, lbl)
        py += PH
    arr(ax, DCX, D_PIPE_TOP+0.08, DCX, D_MOD_Y0, lw=1.2)

    # ── Decoder module ─────────────────────────────────────────────────────
    mod_box(ax, DX0, D_MOD_Y0, DW, D_MOD_H, 'Decoder', nx=True)

    d_rows = [
        'Causal Self-Attn.  (if |p| > 1)',
        'Add & Norm  (if |p| > 1)',
        'Multi-Head Cross Attn.',
        'Add & Norm',
        'Feed Forward',
        'Add & Norm',
    ]
    irw = DW-0.48; ir_x = DX0+0.24
    tot_h = len(d_rows)*IRH + (len(d_rows)-1)*IRG
    ry = D_MOD_Y0 + (D_MOD_H - tot_h)/2
    dirc = []
    for k, lbl in enumerate(d_rows):
        irow(ax, ir_x, ry, irw, IRH, lbl)
        dirc.append(ry+IRH/2)
        if k < len(d_rows)-1:
            arr(ax, DCX, ry+IRH+0.04, DCX, ry+IRH+IRG-0.04,
                lw=1.0, hw=0.09, hl=0.09)
        ry += IRH+IRG

    rx = DX1-0.36
    for yc in [dirc[1], dirc[3], dirc[5]]:
        skip_arrow(ax, rx, yc+IRH*0.44, yc-0.70)

    # ── Head Layer ─────────────────────────────────────────────────────────
    arr(ax, DCX, D_MOD_Y1+0.06, DCX, HEAD_Y0, lw=1.2)
    ax.add_patch(FancyBboxPatch(
        (DX0, HEAD_Y0), DW, HEAD_H,
        boxstyle='round,pad=0.08',
        facecolor=MINT, edgecolor=MINTB, linewidth=1.2, zorder=2))
    txt(ax, DCX, HEAD_Y0+HEAD_H/2, 'Head Layer', fs=14, bold=True, c='#1a5530')

    # ── Output predictions (vertical, going UP from HEAD_Y1) ───────────────
    arr(ax, DCX, HEAD_Y1+0.06, DCX, D_PRED_TOP-VTH/2-0.08, c=RED, lw=1.2)
    pred_toks = dec_toks[nf:] + ([gen] if gen else [])
    pred_cx = DCX + 1.10
    if pred_toks:
        for idx, pt in enumerate(pred_toks):
            cy    = D_PRED_TOP + idx*D_SLOT
            fresh = bool(gen) and idx == len(pred_toks)-1
            vtok(ax, pred_cx, cy, pt, style='pred', fresh=fresh)
        pt_top = D_PRED_TOP + (len(pred_toks)-1)*D_SLOT
        ax.add_patch(FancyBboxPatch(
            (pred_cx-VTW/2-0.16, D_PRED_TOP-VTH/2-0.14),
            VTW+0.32, pt_top-D_PRED_TOP+VTH+0.28,
            boxstyle='round,pad=0.06',
            facecolor='none', edgecolor='#888', linewidth=0.9,
            linestyle='solid', zorder=2))
        txt(ax, DX1+0.16, (D_PRED_TOP+pt_top)/2,
            r'$\hat{y} \in \mathbb{R}^{B \times T \times P}$',
            fs=11, c='#888', ha='left')

    # ── Causal cross-attention fan ─────────────────────────────────────────
    if xattn and enc_out_xs:
        if pred_toks:
            tx = pred_cx
            ty = D_PRED_TOP + (len(pred_toks)-1)*D_SLOT + VTH/2 + 0.10
        else:
            tx = DCX; ty = HEAD_Y0+HEAD_H/2
        N_e = len(enc_out_xs)
        for j, ex in enumerate(enc_out_xs):
            al = max(0.14, 0.92-0.17*j)
            ac = 0.46 - 0.12*(j/max(N_e-1,1))
            arr(ax, ex, E_OUT_Y+HTH/2+0.10, tx, ty,
                c=DARK, lw=0.85 if j<N_e-1 else 1.3,
                arc=ac, alpha=al, hw=0.10, hl=0.10)
        lx = (np.mean(enc_out_xs) + tx) / 2
        ly = E_OUT_Y + HTH/2 + 0.90
        txt(ax, lx, ly, 'Causal cross attention', fs=14, bold=False, c=DARK)

    # ── Feedback arrow ─────────────────────────────────────────────────────
    if gen and dec_all:
        fbx = DX1 + 0.90
        dline(ax,
              [pred_cx+VTW/2, fbx,       fbx,            stack_cx+VTW/2],
              [D_PRED_TOP,    D_PRED_TOP, D_STACK_TOP,    D_STACK_TOP],
              c=RED, lw=1.2)
        ax.annotate('', xy=(stack_cx+VTW/2, D_STACK_TOP),
                    xytext=(stack_cx+VTW/2+0.04, D_STACK_TOP),
                    arrowprops=dict(
                        arrowstyle='->, head_width=0.12, head_length=0.12',
                        color=RED, lw=1.2), zorder=5)
        txt(ax, fbx+0.30, (D_STACK_TOP+D_PRED_TOP)/2,
            'next\nstep', fs=12, c=RED)


# ══════════════════════════════════════════════════════════════════════════
#  LLM Decoder
# ══════════════════════════════════════════════════════════════════════════

def draw_decoder_llm(ax, dec_toks, nf, gen, enc_out_xs):
    dec_all = dec_toks + ([gen] if gen else [])

    # Input word tokens (horizontal)
    if dec_all:
        d_xs = hc(len(dec_all), DCX)
        for i, t in enumerate(dec_all):
            pred  = i >= nf
            fresh = bool(gen) and i == len(dec_all)-1
            htok(ax, d_xs[i], E_IN_Y, t,
                 style='pred' if pred else 'word', fresh=fresh)
        arr(ax, DCX, E_IN_Y+HTH/2+0.08, DCX, E_PIPE_Y0, lw=1.2)
    else:
        txt(ax, DCX, E_IN_Y, '(awaiting start token)', fs=12, c='#aaa')

    txt(ax, DCX, E_IN_Y-0.62, 'Decoder input', fs=12, c='#777')

    # Pipeline
    py = E_PIPE_Y0
    for lbl in ['Embedding', 'Positional Encoding']:
        pipe_blk(ax, DX0+0.12, py, DW-0.24, PH, lbl)
        py += PH
    arr(ax, DCX, E_PIPE_Y0+2*PH+0.08, DCX, LLM_MOD_Y0, lw=1.2)

    # Decoder module
    mod_box(ax, DX0, LLM_MOD_Y0, DW, LLM_MOD_H, 'Decoder', nx=True)
    d_rows = ['Causal Self-Attn.', 'Add & Norm',
              'Multi-Head Cross-Attn.', 'Add & Norm',
              'Feed Forward', 'Add & Norm']
    irw = DW-0.48; ir_x = DX0+0.24
    tot_h = len(d_rows)*IRH + (len(d_rows)-1)*IRG
    ry = LLM_MOD_Y0 + (LLM_MOD_H-tot_h)/2
    dirc = []
    for k, lbl in enumerate(d_rows):
        irow(ax, ir_x, ry, irw, IRH, lbl)
        dirc.append(ry+IRH/2)
        if k < len(d_rows)-1:
            arr(ax, DCX, ry+IRH+0.04, DCX, ry+IRH+IRG-0.04,
                lw=1.0, hw=0.09, hl=0.09)
        ry += IRH+IRG

    rx = DX1-0.36
    for yc in [dirc[1], dirc[3], dirc[5]]:
        skip_arrow(ax, rx, yc+IRH*0.44, yc-0.70)

    # Cross-attention: encoder output → Multi-Head Cross-Attn row
    if dec_all and enc_out_xs:
        ty = dirc[2]
        N_e = len(enc_out_xs)
        for j, ex in enumerate(enc_out_xs):
            al = max(0.14, 0.85-0.15*j)
            ac = 0.28 - 0.06*(j/max(N_e-1,1))
            arr(ax, ex, E_OUT_Y+HTH/2+0.10,
                    DX0-0.10, ty,
                    c=DARK, lw=0.80, arc=ac, alpha=al, hw=0.09, hl=0.09)
        txt(ax, (EX1+DX0)/2,
            (E_OUT_Y+ty)/2+0.45,
            'cross-attention', fs=13, c=DARK)

    # Head Layer
    arr(ax, DCX, LLM_MOD_Y1+0.06, DCX, LLM_HEAD_Y0, lw=1.2)
    ax.add_patch(FancyBboxPatch(
        (DX0, LLM_HEAD_Y0), DW, HEAD_H,
        boxstyle='round,pad=0.08',
        facecolor=MINT, edgecolor=MINTB, linewidth=1.2, zorder=2))
    txt(ax, DCX, LLM_HEAD_Y0+HEAD_H/2, 'Head Layer',
        fs=14, bold=True, c='#1a5530')
    arr(ax, DCX, LLM_HEAD_Y1+0.06, DCX, LLM_OUT_Y-HTH/2-0.08, c=RED, lw=1.2)

    # Output predicted words (horizontal row, dashed red)
    if gen:
        pw = dec_toks[nf:] + [gen]
        p_xs = hc(len(pw), DCX)
        for i, t in enumerate(pw):
            htok(ax, p_xs[i], LLM_OUT_Y, t, style='pred',
                 fresh=(i==len(pw)-1))
        txt(ax, DX1+0.16, LLM_OUT_Y, r'$\hat{y}$', fs=12, c='#888', ha='left')

        # Feedback
        fbx = DX1+0.90
        lrx = p_xs[-1]+HTW/2
        dline(ax, [lrx, fbx, fbx, lrx],
              [LLM_OUT_Y, LLM_OUT_Y, E_IN_Y, E_IN_Y], c=RED)
        ax.annotate('', xy=(lrx, E_IN_Y), xytext=(lrx+0.04, E_IN_Y),
                    arrowprops=dict(
                        arrowstyle='->, head_width=0.12, head_length=0.12',
                        color=RED, lw=1.2), zorder=5)
        txt(ax, fbx+0.30, (E_IN_Y+LLM_OUT_Y)/2, 'next\nstep', fs=12, c=RED)


# ══════════════════════════════════════════════════════════════════════════
#  Animation data
# ══════════════════════════════════════════════════════════════════════════

TSFM_STEPS = [
    dict(desc='Encoder processes all N observed patches using causal self-attention',
         enc_n=5, dec=[], nf=0, gen=None),
    dict(desc='Last patch p\u2085 becomes the start token \u2192 cross-attends encoder \u2192 predicts p\u0302\u2086',
         enc_n=5, dec=['p\u2085'], nf=1, gen='p\u0302\u2086'),
    dict(desc='p\u0302\u2086 fed back into decoder \u2192 cross-attends all encoder patches \u2192 predicts p\u0302\u2087',
         enc_n=5, dec=['p\u2085','p\u0302\u2086'], nf=1, gen='p\u0302\u2087'),
    dict(desc='p\u0302\u2087 appended to decoder context \u2192 predicts p\u0302\u2088',
         enc_n=5, dec=['p\u2085','p\u0302\u2086','p\u0302\u2087'], nf=1, gen='p\u0302\u2088'),
    dict(desc='Autoregressive forecasting continues across the full horizon T',
         enc_n=5, dec=['p\u2085','p\u0302\u2086','p\u0302\u2087','p\u0302\u2088'], nf=1, gen='p\u0302\u2089'),
]

LLM_STEPS = [
    dict(desc='Encoder reads the full input context using causal self-attention',
         enc=['the','cat','sat','on'], dec=[], nf=0, gen=None),
    dict(desc='Decoder receives \u27e8s\u27e9 start token \u2192 cross-attends encoder \u2192 generates W\u2085',
         enc=['the','cat','sat','on'], dec=['\u27e8s\u27e9'], nf=1, gen='W\u2085'),
    dict(desc='W\u2085 fed back \u2192 cross-attends encoder context \u2192 generates W\u2086',
         enc=['the','cat','sat','on'], dec=['\u27e8s\u27e9','W\u2085'], nf=1, gen='W\u2086'),
    dict(desc='W\u2086 appended \u2192 generates W\u2087',
         enc=['the','cat','sat','on'], dec=['\u27e8s\u27e9','W\u2085','W\u2086'], nf=1, gen='W\u2087'),
    dict(desc='Each generated word feeds back as the next decoder input',
         enc=['the','cat','sat','on'],
         dec=['\u27e8s\u27e9','W\u2085','W\u2086','W\u2087'], nf=1, gen='\u27e8/s\u27e9'),
]


# ══════════════════════════════════════════════════════════════════════════
#  Frame renderers
# ══════════════════════════════════════════════════════════════════════════

def divider(ax):
    mx = (EX1+DX0)/2
    ax.plot([mx,mx],[0.7,FH-0.5], color='#dddddd', lw=0.8, ls='--', zorder=1)


def render_tsfm(step, idx):
    fig, ax = new_fig()
    txt(ax, FW/2, FH-0.45, 'TSFM: Autoregressive Patch Prediction',
        fs=17, bold=True, c=NAVY)
    txt(ax, FW/2, FH-1.00, step['desc'], fs=13, c='#444')

    N = step['enc_n']
    enc_out_xs = draw_encoder(ax, [f'p{i+1}' for i in range(N)], is_llm=False)

    # "Start token for decoder" arrow (step 1 only)
    if idx == 1 and step['dec']:
        lx = hc(N, ECX)[-1]
        sx = DCX - 1.20 - VTW/2 - 0.22
        arr(ax, lx+HTW/2+0.08, E_IN_Y, sx, D_STACK_TOP,
            c=DARK, lw=1.4, ls=(0,(5,3)), arc=-0.24, hw=0.13, hl=0.13)
        txt(ax, (lx+sx)/2, E_IN_Y-0.50,
            'Start token for decoder', fs=12, c=DARK)

    draw_decoder_tsfm(ax, step['dec'], step['nf'], step['gen'],
                      enc_out_xs, xattn=(len(step['dec']) > 0))

    txt(ax, ECX, 0.72, 'Encoder', fs=13, c='#555')
    txt(ax, DCX, 0.72, 'Decoder', fs=13, c='#555')
    divider(ax)
    step_dots(ax, len(TSFM_STEPS), idx)
    plt.tight_layout(pad=0.3)
    return fig2pil(fig)


def render_llm(step, idx):
    fig, ax = new_fig()
    txt(ax, FW/2, FH-0.45, 'LLM: Autoregressive Word Generation',
        fs=17, bold=True, c=NAVY)
    txt(ax, FW/2, FH-1.00, step['desc'], fs=13, c='#444')

    enc_out_xs = draw_encoder(ax, step['enc'], is_llm=True)
    draw_decoder_llm(ax, step['dec'], step['nf'], step['gen'], enc_out_xs)

    txt(ax, ECX, 0.72, 'Encoder', fs=13, c='#555')
    txt(ax, DCX, 0.72, 'Decoder', fs=13, c='#555')
    divider(ax)
    step_dots(ax, len(LLM_STEPS), idx)
    plt.tight_layout(pad=0.3)
    return fig2pil(fig)


# ══════════════════════════════════════════════════════════════════════════
#  Main
# ══════════════════════════════════════════════════════════════════════════

def main():
    STEP_MS, PAUSE_MS = 2500, 1500

    print('Rendering LLM frames…')
    lf, ld = [], []
    for i, s in enumerate(LLM_STEPS):
        lf.append(render_llm(s, i))
        ld.append(PAUSE_MS if i==len(LLM_STEPS)-1 else STEP_MS)
    save_gif(lf, 'llm_ar.gif', ld)

    print('Rendering TSFM frames…')
    tf, td = [], []
    for i, s in enumerate(TSFM_STEPS):
        tf.append(render_tsfm(s, i))
        td.append(PAUSE_MS if i==len(TSFM_STEPS)-1 else STEP_MS)
    save_gif(tf, 'tsfm_ar.gif', td)

    print('Building combined GIF…')
    save_gif(lf+tf, 'combined_ar.gif', ld+td)
    print('Done.')


if __name__ == '__main__':
    main()

Rendering LLM frames…
  → llm_ar.gif
Rendering TSFM frames…
  → tsfm_ar.gif
Building combined GIF…
  → combined_ar.gif
Done.
